# PA2 SB-00 to SB-07 Kaggle Verification

This notebook is a clean verification workflow for:

1. Loading and parsing HH-RLHF harmless split into `(prompt, chosen, rejected)` triples.
2. Building and inspecting SFT, RM, and DPO dataloaders.
3. Visual proof that masking is applied correctly for SFT labels.
4. Loading both policy and Llama backbones, and printing parameter + VRAM stats.
5. Verifying reference model isolation (no gradient-sharing state with policy).

Expected runtime: Kaggle GPU (recommended).

In [ ]:
# Uncomment if dependencies are not already installed in your Kaggle runtime.
# If your repo is cloned under /kaggle/working/PA2, this will work directly.
# !pip install -q -r /kaggle/working/PA2/requirements.txt

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/ahmad903501/Deep-Vision-Language-Models-2"
PROJECT_DIR = "/kaggle/working/PA2"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print("Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Project root set to:", os.getcwd())

In [ ]:
import os
import sys
import textwrap
from pprint import pprint

import torch


def find_project_root(start_dir: str) -> str:
    """Find project root by looking for src/ and requirements.txt."""
    cur = os.path.abspath(start_dir)
    for _ in range(5):
        has_src = os.path.isdir(os.path.join(cur, "src"))
        has_req = os.path.isfile(os.path.join(cur, "requirements.txt"))
        if has_src and has_req:
            return cur
        cur = os.path.dirname(cur)
    return os.path.abspath(start_dir)


ROOT = find_project_root(os.getcwd())
if ROOT not in sys.path:
    sys.path.append(ROOT)

print("Project root:", ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from src.config import default_experiment_config
from src.data import build_dataloaders, load_hh_harmless_dataset, parse_hh_split
from src.models.model_factory import create_model_bundle, parameter_counts
from src.utils.memory import MemoryManager

cfg = default_experiment_config()

# You can override via Kaggle env variables if needed.
cfg.model.policy_model_name = os.getenv("POLICY_MODEL", "HuggingFaceTB/SmolLM2-360M")
cfg.model.llama_backbone_name = os.getenv("LLAMA_MODEL", "meta-llama/Llama-3.2-1B-Instruct")
cfg.data.hh_dataset_name = os.getenv("HH_DATASET", "Anthropic/hh-rlhf")
cfg.data.hh_harmless_config = os.getenv("HH_HARMLESS_CONFIG", "harmless-base")

cfg.data.max_seq_len = int(os.getenv("MAX_SEQ_LEN", "1024"))
cfg.data.sft_batch_size = int(os.getenv("SFT_BATCH", "2"))
cfg.data.rm_batch_size = int(os.getenv("RM_BATCH", "2"))
cfg.data.dpo_batch_size = int(os.getenv("DPO_BATCH", "2"))

print("Policy model:", cfg.model.policy_model_name)
print("Llama model:", cfg.model.llama_backbone_name)
print("HH dataset:", cfg.data.hh_dataset_name, "| harmless config:", cfg.data.hh_harmless_config)
print("Max seq len:", cfg.data.max_seq_len)
print("Batch sizes (SFT/RM/DPO):", cfg.data.sft_batch_size, cfg.data.rm_batch_size, cfg.data.dpo_batch_size)

In [ ]:
hh = load_hh_harmless_dataset(
    dataset_name=cfg.data.hh_dataset_name,
    harmless_config=cfg.data.hh_harmless_config,
)

print(hh)
print("Train size:", len(hh["train"]))
print("Test size:", len(hh["test"]))

# Strict parse over both splits: any parse failure should raise immediately.
train_triples = parse_hh_split(hh["train"], skip_invalid=False)
test_triples = parse_hh_split(hh["test"], skip_invalid=False)

print("\n=== PARSE REPORT ===")
print("Parsed train triples:", len(train_triples))
print("Parsed test triples:", len(test_triples))
print("Skipped rows: 0 (strict parse mode)")

In [ ]:
print("=== 3 Parsed Triples (FULL prompt/context + chosen + rejected) ===")

for i, tri in enumerate(train_triples[:3], start=1):
    print(f"\n{'=' * 120}")
    print(f"TRIPLE {i}")
    print(f"{'=' * 120}")

    print("\n[Prompt / Context]")
    print(tri.prompt)

    print("\n[Chosen Response]")
    print(tri.chosen)

    print("\n[Rejected Response]")
    print(tri.rejected)

    print("\n[Sanity Checks]")
    print("Prompt contains assistant marker:", "Assistant:" in tri.prompt)
    print("Chosen raw length:", len(tri.chosen))
    print("Rejected raw length:", len(tri.rejected))

In [ ]:
# This loads:
# 1) policy model (SmolLM + LoRA)
# 2) reference model (separate frozen copy)
# 3) llama backbone model
# and both tokenizers with required padding conventions.
bundle = create_model_bundle(cfg)

print("Policy tokenizer padding_side:", bundle.policy_tokenizer.padding_side)
print("RM tokenizer padding_side:", bundle.rm_tokenizer.padding_side)
print("Policy pad_token_id == eos_token_id:", bundle.policy_tokenizer.pad_token_id == bundle.policy_tokenizer.eos_token_id)
print("RM pad_token_id == eos_token_id:", bundle.rm_tokenizer.pad_token_id == bundle.rm_tokenizer.eos_token_id)

assert bundle.policy_tokenizer.pad_token_id == bundle.policy_tokenizer.eos_token_id
assert bundle.rm_tokenizer.pad_token_id == bundle.rm_tokenizer.eos_token_id
print("Pad token is EOS for both tokenizers: PASS")

In [ ]:
# Reference model must share no gradient state with policy model.
ref_all_frozen = all(not p.requires_grad for p in bundle.reference_model.parameters())
policy_first = next(bundle.policy_model.parameters())
ref_first = next(bundle.reference_model.parameters())

print("Reference fully frozen:", ref_all_frozen)
print("Policy/reference first tensor data_ptr differs:", policy_first.data_ptr() != ref_first.data_ptr())

assert ref_all_frozen, "Reference model has trainable params, which violates isolation."
assert policy_first.data_ptr() != ref_first.data_ptr(), "Policy and reference appear to share storage."
print("Reference isolation checks: PASS")

In [ ]:
subset_n = max(cfg.data.sft_batch_size, cfg.data.rm_batch_size, cfg.data.dpo_batch_size) * 3
subset = train_triples[:subset_n]

loaders = build_dataloaders(
    triples=subset,
    policy_tokenizer=bundle.policy_tokenizer,
    rm_tokenizer=bundle.rm_tokenizer,
    data_config=cfg.data,
)

print("=== Example from each dataloader: SFT / RM / DPO ===")
for name in ["sft", "rm", "dpo"]:
    batch = next(iter(loaders[name]))
    print(f"\n[{name.upper()}]")
    for k, v in batch.items():
        if torch.is_tensor(v):
            print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")
        elif isinstance(v, list):
            sample = str(v[0]).replace("\n", " ")[:120]
            print(f"  {k}: list(len={len(v)}), sample='{sample}...' ")
        else:
            print(f"  {k}: {type(v)}")

In [ ]:
# Visual masking check on one SFT sample.
sft_batch = next(iter(loaders["sft"]))
row = 0

input_ids = sft_batch["input_ids"][row]
labels = sft_batch["labels"][row]
attn = sft_batch["attention_mask"][row]
response_mask = sft_batch["response_mask"][row]

tokens = bundle.policy_tokenizer.convert_ids_to_tokens(input_ids.tolist())

print("=== Masking Visualization (non-pad tokens only) ===")
print("pos | token                          | label   | response_mask | contributes_to_loss")
print("-" * 95)

for pos, (tok, lbl, am, rm) in enumerate(zip(tokens, labels.tolist(), attn.tolist(), response_mask.tolist())):
    if am == 0:
        continue
    contributes = (lbl != -100)
    clean = tok.replace("\n", "\\n").replace("▁", " ").replace("Ġ", " ")
    print(f"{pos:03d} | {clean[:30]:30s} | {lbl:7d} | {rm:13d} | {contributes}")

print("\nSummary:")
print("Valid (non-pad) tokens:", int(attn.sum().item()))
print("Tokens contributing to CE loss:", int((labels != -100).sum().item()))
print("Prompt-masked tokens:", int(((attn == 1) & (labels == -100)).sum().item()))

In [ ]:
def human_count(n: int) -> str:
    if n >= 1_000_000_000:
        return f"{n / 1_000_000_000:.2f}B"
    if n >= 1_000_000:
        return f"{n / 1_000_000:.2f}M"
    return str(n)

policy_total, policy_trainable = parameter_counts(bundle.policy_model)
llama_total, llama_trainable = parameter_counts(bundle.llama_backbone_model)

print("=== Parameter Counts ===")
print("Policy total/trainable:", human_count(policy_total), "/", human_count(policy_trainable))
print("Llama total/trainable:", human_count(llama_total), "/", human_count(llama_trainable))

print("\n=== GPU Memory Footprint (torch.cuda.memory_allocated) ===")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory
    total_vram_mb = total_vram / (1024 ** 2)

    base = torch.cuda.memory_allocated()
    bundle.policy_model.to("cuda")
    policy_bytes = torch.cuda.memory_allocated() - base

    after_policy = torch.cuda.memory_allocated()
    bundle.llama_backbone_model.to("cuda")
    llama_bytes = torch.cuda.memory_allocated() - after_policy

    combined = torch.cuda.memory_allocated()

    print(f"Policy allocated: {policy_bytes / (1024 ** 2):.2f} MB")
    print(f"Llama allocated: {llama_bytes / (1024 ** 2):.2f} MB")
    print(f"Combined allocated: {combined / (1024 ** 2):.2f} MB / {total_vram_mb:.2f} MB total")

    fits = combined <= total_vram * 0.95
    print("Both models fit in available VRAM (95% threshold):", fits)

    # Keep reference model offloaded by default.
    mm = MemoryManager()
    mm.offload_to_cpu(bundle.reference_model)

    # Cleanup back to CPU to avoid holding GPU memory after verification.
    bundle.policy_model.to("cpu")
    bundle.llama_backbone_model.to("cpu")
    torch.cuda.empty_cache()
else:
    print("CUDA not available in this runtime, cannot compute GPU allocation.")

In [ ]:
# Full dataset parser audit: verify every chosen/rejected row has a parseable final assistant segment.
# This scans both train and test splits and can take a few minutes.

from src.data.hh_parser import split_prompt_response

if "hh" not in globals():
    raise RuntimeError("Dataset is not loaded. Run Cell 6 first.")

error_count = 0

for split_name in ["train", "test"]:
    split_ds = hh[split_name]
    print(f"Scanning {split_name} split with {len(split_ds)} rows...")

    for idx, row in enumerate(split_ds):
        for field in ["chosen", "rejected"]:
            try:
                _prompt, _response = split_prompt_response(row[field])
            except ValueError:
                error_count += 1

        if idx > 0 and idx % 10000 == 0:
            print(f"  Processed {idx} rows in {split_name}...")

full_parse_audit_error_count = error_count
full_parse_audit_ok = error_count == 0

print("\n=== Full Parse Audit Result ===")
print("Rows with parse failures:", full_parse_audit_error_count)

assert full_parse_audit_ok, "Found rows with parse failures in chosen/rejected fields."

In [ ]:
# Final consolidated PASS/FAIL summary.

summary = []


def add_result(name: str, ok: bool, detail: str = ""):
    summary.append((name, bool(ok), detail))


# 1) Parsed triples check.
try:
    triples_ok = (
        "train_triples" in globals()
        and len(train_triples) >= 3
        and all(train_triples[i].prompt.endswith("Assistant:") for i in range(3))
        and all(isinstance(train_triples[i].chosen, str) for i in range(3))
        and all(isinstance(train_triples[i].rejected, str) for i in range(3))
    )
    add_result(
        "Parsed 3 valid triples",
        triples_ok,
        "Prompt must end with Assistant: and responses may be empty but must be strings.",
    )
except Exception as exc:  # noqa: BLE001
    add_result("Parsed 3 valid triples", False, f"Exception: {exc}")


# 2) Dataloader examples check.
try:
    loaders_ok = (
        "loaders" in globals()
        and all(k in loaders for k in ["sft", "rm", "dpo"])
    )
    if loaders_ok:
        _ = next(iter(loaders["sft"]))
        _ = next(iter(loaders["rm"]))
        _ = next(iter(loaders["dpo"]))
    add_result("SFT/RM/DPO dataloaders available", loaders_ok, "Expected keys: sft, rm, dpo.")
except Exception as exc:  # noqa: BLE001
    add_result("SFT/RM/DPO dataloaders available", False, f"Exception: {exc}")


# 3) Masking correctness check.
try:
    masking_ok = False
    if "sft_batch" in globals():
        labels = sft_batch["labels"]
        attn = sft_batch["attention_mask"]
        response_mask = sft_batch["response_mask"]

        pad_ignored_ok = bool(torch.all(labels[attn == 0] == -100).item()) if (attn == 0).any() else True
        response_mask_match = bool(torch.equal(response_mask, (labels != -100).long()))
        has_loss_tokens = bool((labels != -100).any().item())

        masking_ok = pad_ignored_ok and response_mask_match and has_loss_tokens

    add_result(
        "SFT masking is consistent",
        masking_ok,
        "Checks pad-ignore, response_mask alignment, and non-empty loss tokens.",
    )
except Exception as exc:  # noqa: BLE001
    add_result("SFT masking is consistent", False, f"Exception: {exc}")


# 4) Pad token = EOS check.
try:
    pad_ok = (
        "bundle" in globals()
        and bundle.policy_tokenizer.pad_token_id == bundle.policy_tokenizer.eos_token_id
        and bundle.rm_tokenizer.pad_token_id == bundle.rm_tokenizer.eos_token_id
    )
    add_result("Pad token equals EOS token", pad_ok, "Required for both policy and RM tokenizers.")
except Exception as exc:  # noqa: BLE001
    add_result("Pad token equals EOS token", False, f"Exception: {exc}")


# 5) Reference model isolation check.
try:
    ref_ok = False
    if "bundle" in globals():
        ref_all_frozen = all(not p.requires_grad for p in bundle.reference_model.parameters())
        policy_first = next(bundle.policy_model.parameters())
        ref_first = next(bundle.reference_model.parameters())
        no_shared_storage = policy_first.data_ptr() != ref_first.data_ptr()
        ref_ok = ref_all_frozen and no_shared_storage
    add_result("Reference model isolation", ref_ok, "Frozen params + independent tensor storage.")
except Exception as exc:  # noqa: BLE001
    add_result("Reference model isolation", False, f"Exception: {exc}")


# 6) Model counts and VRAM fit check.
try:
    counts_ok = ("policy_total" in globals() and "llama_total" in globals() and policy_total > 0 and llama_total > 0)

    if torch.cuda.is_available():
        vram_ok = ("fits" in globals() and bool(fits))
        add_result("Model params counted", counts_ok, "Policy and Llama param counts must be computed.")
        add_result("VRAM fit check", vram_ok, "Expected True from combined allocation threshold check.")
    else:
        add_result("Model params counted", counts_ok, "Policy and Llama param counts must be computed.")
        add_result("VRAM fit check", False, "CUDA not available in this runtime.")
except Exception as exc:  # noqa: BLE001
    add_result("Model params counted", False, f"Exception: {exc}")
    add_result("VRAM fit check", False, f"Exception: {exc}")


# 7) Full dataset parse audit check.
try:
    if "full_parse_audit_ok" in globals():
        audit_ok = bool(full_parse_audit_ok)
        audit_errors = int(globals().get("full_parse_audit_error_count", -1))
        add_result(
            "Full HH train+test parse audit",
            audit_ok,
            f"Rows with missing assistant marker parse failures: {audit_errors}",
        )
    else:
        add_result(
            "Full HH train+test parse audit",
            False,
            "Run Cell 13 to execute full-dataset audit.",
        )
except Exception as exc:  # noqa: BLE001
    add_result("Full HH train+test parse audit", False, f"Exception: {exc}")


# Print summary.
all_pass = all(item[1] for item in summary)

print("=" * 92)
print("FINAL VERIFICATION SUMMARY")
print("=" * 92)
for idx, (name, ok, detail) in enumerate(summary, start=1):
    status = "PASS" if ok else "FAIL"
    print(f"{idx:02d}. [{status}] {name}")
    if detail:
        print(f"    {detail}")

print("-" * 92)
print("OVERALL:", "PASS" if all_pass else "FAIL")

if not all_pass:
    print("\nTip: run all cells from top to bottom, then re-run this summary cell.")

## SB10-SB15 Training Section (RM + SFT)

Use this section to actually train:

1. Reward model (SB10-SB12) with `adaptation_mode` = `"lora"` or `"head_only"`.
2. SFT warm-up (SB13-SB15), including perplexity, sample generations, and checkpoint artifacts.

Recommended in Kaggle for quick sanity: keep dataset limits small first, then increase.

In [ ]:
# Training configuration for SB10-SB15.
# Tune these values before running RM/SFT cells below.

RUN_RM = True
RUN_SFT = True

# For quick bring-up in Kaggle, start small; for full run set to None.
TRAIN_LIMIT = 2000
TEST_LIMIT = 400

RM_ADAPTATION_MODE = "lora"      # "lora" or "head_only"
RM_EPOCHS = 1
RM_LR = 1e-5

SFT_EPOCHS = 1
SFT_LR = 2e-5
SFT_GRAD_ACCUM = 4

ARTIFACT_BASE_DIR = "artifacts"

print("RUN_RM:", RUN_RM)
print("RUN_SFT:", RUN_SFT)
print("TRAIN_LIMIT:", TRAIN_LIMIT, "| TEST_LIMIT:", TEST_LIMIT)
print("RM_ADAPTATION_MODE:", RM_ADAPTATION_MODE)
print("Artifact dir:", ARTIFACT_BASE_DIR)

In [ ]:
# SB10-SB12: Reward model training + evaluation + artifact save

import os

import torch

from src.data import build_dataloaders, load_hh_harmless_dataset, parse_hh_split
from src.models import create_model_bundle, load_reward_model
from src.trainers import evaluate_reward_model, save_reward_artifact, train_reward_model

if RUN_RM:
    cfg_rm = default_experiment_config()
    cfg_rm.rm_train.adaptation_mode = RM_ADAPTATION_MODE
    cfg_rm.rm_train.epochs = RM_EPOCHS
    cfg_rm.rm_train.learning_rate = RM_LR

    print("\n[RM] Loading dataset...")
    ds = load_hh_harmless_dataset(
        dataset_name=cfg_rm.data.hh_dataset_name,
        harmless_config=cfg_rm.data.hh_harmless_config,
    )

    train_triples_rm = parse_hh_split(ds["train"], limit=TRAIN_LIMIT, skip_invalid=False)
    test_triples_rm = parse_hh_split(ds["test"], limit=TEST_LIMIT, skip_invalid=False)

    print(f"[RM] train triples: {len(train_triples_rm)} | test triples: {len(test_triples_rm)}")

    print("[RM] Loading models/tokenizers...")
    bundle_rm = create_model_bundle(cfg_rm)

    rm_train_loaders = build_dataloaders(
        train_triples_rm,
        bundle_rm.policy_tokenizer,
        bundle_rm.rm_tokenizer,
        cfg_rm.data,
    )
    rm_test_loaders = build_dataloaders(
        test_triples_rm,
        bundle_rm.policy_tokenizer,
        bundle_rm.rm_tokenizer,
        cfg_rm.data,
    )

    rm_model = load_reward_model(cfg_rm, rm_tokenizer=bundle_rm.rm_tokenizer)

    # Required check from assignment: RM pad_token_id must be set correctly.
    print("[RM] rm.config.pad_token_id:", rm_model.config.pad_token_id)
    print("[RM] rm tokenizer pad_token_id:", bundle_rm.rm_tokenizer.pad_token_id)
    assert rm_model.config.pad_token_id == bundle_rm.rm_tokenizer.pad_token_id

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("[RM] Training...")
    rm_train_metrics = train_reward_model(rm_model, rm_train_loaders["rm"], cfg_rm, device=device)

    print("[RM] Evaluating...")
    rm_eval_metrics = evaluate_reward_model(rm_model, rm_test_loaders["rm"], device=device)

    rm_out_dir = os.path.join(ARTIFACT_BASE_DIR, "rm_model")
    rm_saved_path = save_reward_artifact(rm_model, bundle_rm.rm_tokenizer, rm_out_dir)

    print("\n=== RM TRAIN METRICS ===")
    print(rm_train_metrics)
    print("\n=== RM EVAL METRICS ===")
    print(rm_eval_metrics)
    print("\nRM adaptation mode:", cfg_rm.rm_train.adaptation_mode)
    print("Saved RM artifact:", rm_saved_path)
else:
    print("RUN_RM is False, skipping reward model training cell.")

In [ ]:
# SB13-SB15: SFT training + eval + sample generation + artifact save

import os

import torch

from src.data import build_dataloaders, load_hh_harmless_dataset, parse_hh_split
from src.models import create_model_bundle
from src.trainers import (
    evaluate_sft_perplexity,
    generate_sft_samples,
    save_sft_artifacts,
    train_sft,
)

if RUN_SFT:
    cfg_sft = default_experiment_config()
    cfg_sft.sft_train.epochs = SFT_EPOCHS
    cfg_sft.sft_train.learning_rate = SFT_LR
    cfg_sft.sft_train.grad_accum_steps = SFT_GRAD_ACCUM

    print("\n[SFT] Loading dataset...")
    ds = load_hh_harmless_dataset(
        dataset_name=cfg_sft.data.hh_dataset_name,
        harmless_config=cfg_sft.data.hh_harmless_config,
    )

    train_triples_sft = parse_hh_split(ds["train"], limit=TRAIN_LIMIT, skip_invalid=False)
    test_triples_sft = parse_hh_split(ds["test"], limit=TEST_LIMIT, skip_invalid=False)

    print(f"[SFT] train triples: {len(train_triples_sft)} | test triples: {len(test_triples_sft)}")

    print("[SFT] Loading models/tokenizers...")
    bundle_sft = create_model_bundle(cfg_sft)

    sft_train_loaders = build_dataloaders(
        train_triples_sft,
        bundle_sft.policy_tokenizer,
        bundle_sft.rm_tokenizer,
        cfg_sft.data,
    )
    sft_test_loaders = build_dataloaders(
        test_triples_sft,
        bundle_sft.policy_tokenizer,
        bundle_sft.rm_tokenizer,
        cfg_sft.data,
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("[SFT] Training...")
    sft_train_metrics = train_sft(bundle_sft.policy_model, sft_train_loaders["sft"], cfg_sft, device=device)

    print("[SFT] Evaluating perplexity...")
    sft_ppl = evaluate_sft_perplexity(
        bundle_sft.policy_model,
        sft_test_loaders["sft"],
        device=device,
        max_batches=20,
    )

    print("[SFT] Generating sample outputs...")
    sample_prompts = [
        "\n\nHuman: Explain why regular exercise helps long-term health.\n\nAssistant:",
        "\n\nHuman: Give me 3 ideas to study more consistently this week.\n\nAssistant:",
    ]
    samples = generate_sft_samples(
        bundle_sft.policy_model,
        bundle_sft.policy_tokenizer,
        sample_prompts,
        device=device,
        max_new_tokens=cfg_sft.sft_train.sample_gen_max_new_tokens,
    )

    sft_out_dir = os.path.join(ARTIFACT_BASE_DIR, "sft")
    sft_paths = save_sft_artifacts(
        policy_model=bundle_sft.policy_model,
        tokenizer=bundle_sft.policy_tokenizer,
        output_dir=sft_out_dir,
        base_checkpoint_name=cfg_sft.model.policy_model_name,
    )

    print("\n=== SFT TRAIN METRICS ===")
    print(sft_train_metrics)
    print("Held-out perplexity:", sft_ppl)

    print("\n=== SAMPLE GENERATIONS ===")
    for i, text in enumerate(samples):
        print(f"\nSample {i + 1}:\n{text}")

    print("\nSaved SFT artifacts:")
    print(sft_paths)
else:
    print("RUN_SFT is False, skipping SFT training cell.")